# Multi-frequency DEC visitation modeling

This notebook trains and tests separate forecasting models for hourly, daily, monthly, and yearly visitation. It follows the State_model convention of chronological evaluation: later observations are held out, never randomly mixed into training.

Models compared:
- previous-period baseline;
- seasonal baseline (24 hours, 7 days, 12 months, or 1 year);
- regularized ridge regression on calendar, location, lag, and rolling-history features.

Lag features use the previous observed value, so results represent one-step-ahead forecasting. The notebook uses only NumPy, Pandas, and Matplotlib.

In [21]:
from pathlib import Path
import json, warnings, sys
import numpy as np
import pandas as pd
NOTEBOOK_DIR = Path.cwd()
PROJECT_ROOT = NOTEBOOK_DIR.parent if NOTEBOOK_DIR.name == 'final code' else NOTEBOOK_DIR
LOCAL_ML_PACKAGES = PROJECT_ROOT / 'final code' / '.ml_packages'
if LOCAL_ML_PACKAGES.exists(): sys.path.insert(0,str(LOCAL_ML_PACKAGES))
try:
    import matplotlib.pyplot as plt
    HAS_MATPLOTLIB = True
except ImportError:
    plt = None
    HAS_MATPLOTLIB = False

warnings.filterwarnings('ignore')
DATA_DIR = PROJECT_ROOT / 'data'
OUTPUT_DIR = DATA_DIR / 'ml_outputs'
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
RANDOM_STATE = 42

CONFIG = {
    'hourly': {'file':'hourly.csv','lags':[1,24,168],'rolls':[24,168],'seasonal_lag':24,'min_rows':500},
    'daily': {'file':'daily.csv','lags':[1,7,28],'rolls':[7,28],'seasonal_lag':7,'min_rows':90},
    'monthly': {'file':'monthly.csv','lags':[1,3,12],'rolls':[3,12],'seasonal_lag':12,'min_rows':24},
    'yearly': {'file':'yearly.csv','lags':[1,2,3],'rolls':[2,3],'seasonal_lag':1,'min_rows':6},
}
print('Input folder:',DATA_DIR)
print('Output folder:',OUTPUT_DIR)

Input folder: /Users/ineshvytheswaran/iCloud Drive (Archive)/Desktop/Desktop - MacBook Air (407)/CSC fellowship/Final_CSC_Workspace/data
Output folder: /Users/ineshvytheswaran/iCloud Drive (Archive)/Desktop/Desktop - MacBook Air (407)/CSC fellowship/Final_CSC_Workspace/data/ml_outputs


In [22]:
def load_frequency(name):
    path = DATA_DIR / CONFIG[name]['file']
    df = pd.read_csv(path)
    df['start_date'] = pd.to_datetime(df.start_date,errors='coerce')
    df['visitation_count'] = pd.to_numeric(df.visitation_count,errors='coerce')
    df = df.dropna(subset=['location','start_date','visitation_count']).query('visitation_count >= 0').copy()
    return df.sort_values(['location','start_date']).reset_index(drop=True)

profiles = []
for name in CONFIG:
    df = load_frequency(name)
    profiles.append({'frequency':name,'records':len(df),'locations':df.location.nunique(),'start':df.start_date.min(),'end':df.start_date.max(),'mean_visitation':df.visitation_count.mean()})
profile_df = pd.DataFrame(profiles)
display(profile_df)

,frequency,records,locations,start,end,mean_visitation
0,hourly,49033,1,2020-07-22 12:00:00,2026-04-01 13:00:00,2.981359
1,daily,2051,1,2020-07-22 00:00:00,2026-04-01 00:00:00,71.274988
2,monthly,21268,103,1991-01-01 00:00:00,2026-07-01 00:00:00,131.088447
3,yearly,2150,103,1991-01-01 00:00:00,2026-01-01 00:00:00,1357.944226


In [23]:
def add_features(df,name):
    cfg = CONFIG[name]
    out = df.sort_values(['location','start_date']).copy()
    grouped = out.groupby('location',group_keys=False)['visitation_count']
    for lag in cfg['lags']:
        out[f'lag_{lag}'] = grouped.shift(lag)
    for window in cfg['rolls']:
        out[f'rolling_mean_{window}'] = grouped.transform(lambda s: s.shift(1).rolling(window,min_periods=max(2,window//3)).mean())
    out['year_centered'] = out.start_date.dt.year - out.start_date.dt.year.min()
    out['month_sin'] = np.sin(2*np.pi*out.start_date.dt.month/12)
    out['month_cos'] = np.cos(2*np.pi*out.start_date.dt.month/12)
    out['dow_sin'] = np.sin(2*np.pi*out.start_date.dt.dayofweek/7)
    out['dow_cos'] = np.cos(2*np.pi*out.start_date.dt.dayofweek/7)
    out['hour_sin'] = np.sin(2*np.pi*out.start_date.dt.hour/24)
    out['hour_cos'] = np.cos(2*np.pi*out.start_date.dt.hour/24)
    out['time_index'] = out.groupby('location').cumcount()
    feature_cols = ['year_centered','month_sin','month_cos','dow_sin','dow_cos','hour_sin','hour_cos','time_index'] + [f'lag_{x}' for x in cfg['lags']] + [f'rolling_mean_{x}' for x in cfg['rolls']]
    out = out.dropna(subset=feature_cols+['visitation_count']).copy()
    eligible = out.groupby('location').size()
    eligible = eligible[eligible >= cfg['min_rows']].index
    out = out[out.location.isin(eligible)].copy()
    return out, feature_cols

def chronological_split(df,test_fraction=0.20):
    parts=[]
    for location,g in df.groupby('location',sort=False):
        g=g.sort_values('start_date').copy()
        n_test=max(1,int(np.ceil(len(g)*test_fraction)))
        g['split']='train'; g.iloc[-n_test:,g.columns.get_loc('split')]='test'
        parts.append(g)
    return pd.concat(parts).sort_values(['location','start_date']).reset_index(drop=True)

def design_matrix(frame,feature_cols,train_mask):
    numeric=frame[feature_cols].astype(float).copy()
    means=numeric.loc[train_mask].mean(); stds=numeric.loc[train_mask].std().replace(0,1).fillna(1)
    numeric=(numeric-means)/stds
    locations=pd.get_dummies(frame.location,prefix='location',dtype=float)
    X=np.column_stack([np.ones(len(frame)),numeric.to_numpy(),locations.to_numpy()])
    names=['intercept']+feature_cols+locations.columns.tolist()
    return X,names

def ridge_fit(X,y,alpha):
    penalty=np.eye(X.shape[1])*alpha; penalty[0,0]=0
    return np.linalg.solve(X.T@X+penalty,X.T@y)

def metrics(y,p):
    y=np.asarray(y,float); p=np.asarray(p,float); mask=np.isfinite(y)&np.isfinite(p); y,p=y[mask],p[mask]
    if not len(y): return {'n':0,'MAE':np.nan,'RMSE':np.nan,'R2':np.nan,'sMAPE_pct':np.nan,'bias':np.nan}
    mae=np.mean(np.abs(y-p)); rmse=np.sqrt(np.mean((y-p)**2)); denom=np.sum((y-y.mean())**2)
    r2=1-np.sum((y-p)**2)/denom if denom>0 else np.nan
    smape=100*np.mean(2*np.abs(p-y)/(np.abs(y)+np.abs(p)+1e-9))
    return {'n':len(y),'MAE':mae,'RMSE':rmse,'R2':r2,'sMAPE_pct':smape,'bias':np.mean(p-y)}

In [24]:
def run_frequency_model(name):
    raw=load_frequency(name)
    model_data,feature_cols=add_features(raw,name)
    if model_data.empty: raise ValueError(f'No eligible rows for {name}')
    model_data=chronological_split(model_data)
    train_mask=model_data.split.eq('train').to_numpy(); test_mask=~train_mask
    X,feature_names=design_matrix(model_data,feature_cols,train_mask)
    y=np.log1p(model_data.visitation_count.to_numpy(float))

    # Tune alpha using only the final 20% of each training location as validation.
    validation=np.zeros(len(model_data),dtype=bool)
    for location,idx in model_data[model_data.split.eq('train')].groupby('location').groups.items():
        ordered=np.array(list(idx)); n=max(1,int(np.ceil(len(ordered)*0.20))); validation[ordered[-n:]]=True
    subtrain=train_mask & ~validation
    alpha_scores=[]
    for alpha in [0.1,1.0,10.0,100.0,1000.0]:
        coef=ridge_fit(X[subtrain],y[subtrain],alpha)
        pred=np.maximum(0,np.expm1(X[validation]@coef))
        alpha_scores.append((alpha,metrics(model_data.loc[validation,'visitation_count'],pred)['RMSE']))
    best_alpha=min(alpha_scores,key=lambda x:x[1])[0]
    coef=ridge_fit(X[train_mask],y[train_mask],best_alpha)
    ridge_pred=np.maximum(0,np.expm1(X[test_mask]@coef))

    test=model_data.loc[test_mask].copy()
    test['observed']=test.visitation_count
    test['ridge_prediction']=ridge_pred
    test['previous_prediction']=test['lag_1']
    test['seasonal_prediction']=test[f"lag_{CONFIG[name]['seasonal_lag']}"]
    result_rows=[]
    for model,col in [('Previous observation','previous_prediction'),('Seasonal naive','seasonal_prediction'),('Ridge calendar + history','ridge_prediction')]:
        result_rows.append({'frequency':name,'model':model,'best_alpha':best_alpha if model.startswith('Ridge') else np.nan,'train_rows':int(train_mask.sum()),'test_rows':int(test_mask.sum()),'eligible_locations':model_data.location.nunique(),**metrics(test.observed,test[col])})
    results=pd.DataFrame(result_rows)
    keep=['location','start_date','observed','previous_prediction','seasonal_prediction','ridge_prediction','split']
    test[keep].to_csv(OUTPUT_DIR/f'{name}_test_predictions.csv',index=False)

    # Plot the eligible location with the most held-out observations.
    plot_location=test.location.value_counts().index[0]
    plot_df=test[test.location.eq(plot_location)].sort_values('start_date').tail(500)
    if HAS_MATPLOTLIB:
        fig,ax=plt.subplots(figsize=(13,5))
        ax.plot(plot_df.start_date,plot_df.observed,label='Observed',linewidth=1.8)
        ax.plot(plot_df.start_date,plot_df.ridge_prediction,label='Ridge prediction',linewidth=1.3)
        ax.set_title(f'{name.title()} held-out forecast: {plot_location}')
        ax.set_ylabel('Visitation count'); ax.legend(); ax.grid(alpha=.25); fig.tight_layout()
        fig.savefig(OUTPUT_DIR/f'{name}_observed_vs_predicted.png',dpi=160,bbox_inches='tight'); plt.close(fig)
    return results,test,best_alpha,alpha_scores

all_results=[]; prediction_tables={}
for frequency in CONFIG:
    print(f'\nTraining {frequency} model...')
    results,predictions,best_alpha,alpha_scores=run_frequency_model(frequency)
    all_results.append(results); prediction_tables[frequency]=predictions
    print('Alpha validation RMSE:',alpha_scores)
    display(results)
results_df=pd.concat(all_results,ignore_index=True)
results_df.to_csv(OUTPUT_DIR/'multi_frequency_model_results.csv',index=False)


Training hourly model...
Alpha validation RMSE: [(0.1, 5.288463068689551), (1.0, 5.2881123909297205), (10.0, 5.284613625540991), (100.0, 5.2503635666758575), (1000.0, 4.963155350868462)]


,frequency,model,best_alpha,train_rows,test_rows,eligible_locations,n,MAE,RMSE,R2,sMAPE_pct,bias
0,hourly,Previous observation,NaN,39092,9773,1,9773,1.490433,3.536192,0.454308,51.200522,0.000000
1,hourly,Seasonal naive,NaN,39092,9773,1,9773,2.026604,4.859933,-0.030710,60.166441,0.000205
2,hourly,Ridge calendar + history,1000.0,39092,9773,1,9773,1.461680,6.148862,-0.649929,107.674636,-0.481009



Training daily model...
Alpha validation RMSE: [(0.1, 44.56742524633038), (1.0, 44.58809245092156), (10.0, 44.70397734779604), (100.0, 45.06459409949812), (1000.0, 48.24945314454622)]


,frequency,model,best_alpha,train_rows,test_rows,eligible_locations,n,MAE,RMSE,R2,sMAPE_pct,bias
0,daily,Previous observation,NaN,1618,405,1,405,33.059259,51.559938,0.052488,80.593933,0.046914
1,daily,Seasonal naive,NaN,1618,405,1,405,28.750617,42.732357,0.349161,78.940045,-0.217284
2,daily,Ridge calendar + history,0.1,1618,405,1,405,22.499743,36.155625,0.534079,61.469385,-11.478597



Training monthly model...
Alpha validation RMSE: [(0.1, 1925.942049359067), (1.0, 2093.0896334210156), (10.0, 3513.2532360782143), (100.0, 14651.521160373943), (1000.0, 99241.43242632553)]


,frequency,model,best_alpha,train_rows,test_rows,eligible_locations,n,MAE,RMSE,R2,sMAPE_pct,bias
0,monthly,Previous observation,NaN,15979,4034,83,4034,66.848656,158.419125,0.669549,50.372451,0.833538
1,monthly,Seasonal naive,NaN,15979,4034,83,4034,63.003726,151.348160,0.698390,47.058644,6.015877
2,monthly,Ridge calendar + history,0.1,15979,4034,83,4034,149.802260,1205.087633,-18.121786,49.766329,88.714578



Training yearly model...
Alpha validation RMSE: [(0.1, 3034.546117402696), (1.0, 5004.560795655804), (10.0, 14348.07469830813), (100.0, 34273.160983571834), (1000.0, 10315.93276352549)]


,frequency,model,best_alpha,train_rows,test_rows,eligible_locations,n,MAE,RMSE,R2,sMAPE_pct,bias
0,yearly,Previous observation,NaN,1441,396,93,396,627.433364,1327.697947,0.623558,47.572578,271.171562
1,yearly,Seasonal naive,NaN,1441,396,93,396,627.433364,1327.697947,0.623558,47.572578,271.171562
2,yearly,Ridge calendar + history,0.1,1441,396,93,396,1832.955002,8526.171048,-14.524149,49.658074,1512.741325


In [25]:
best_models=(results_df.sort_values(['frequency','RMSE']).groupby('frequency',as_index=False).first())
print('Best held-out model by frequency:')
display(best_models[['frequency','model','n','MAE','RMSE','R2','sMAPE_pct','bias']])

coverage=[]
for frequency,pred in prediction_tables.items():
    coverage.append({'frequency':frequency,'test_rows':len(pred),'test_locations':pred.location.nunique(),'test_start':pred.start_date.min(),'test_end':pred.start_date.max()})
coverage_df=pd.DataFrame(coverage)
coverage_df.to_csv(OUTPUT_DIR/'test_coverage_summary.csv',index=False)
display(coverage_df)
print('Saved outputs:')
for path in sorted(OUTPUT_DIR.iterdir()): print('-',path.name)

Best held-out model by frequency:


,frequency,model,n,MAE,RMSE,R2,sMAPE_pct,bias
0,daily,Ridge calendar + history,405,22.499743,36.155625,0.534079,61.469385,-11.478597
1,hourly,Previous observation,9773,1.490433,3.536192,0.454308,51.200522,0.000000
2,monthly,Seasonal naive,4034,63.003726,151.348160,0.698390,47.058644,6.015877
3,yearly,Previous observation,396,627.433364,1327.697947,0.623558,47.572578,271.171562


,frequency,test_rows,test_locations,test_start,test_end
0,hourly,9773,1,2025-02-11 19:00:00,2026-04-01 13:00:00
1,daily,405,1,2025-02-17 00:00:00,2026-04-01 00:00:00
2,monthly,4034,83,2015-06-01 00:00:00,2026-07-01 00:00:00
3,yearly,396,93,2015-01-01 00:00:00,2026-01-01 00:00:00


Saved outputs:
- daily_best_strava_model.png
- daily_observed_vs_predicted.png
- daily_test_predictions.csv
- elm_ridge_edge_model_predictions.csv
- elm_ridge_edge_model_results.csv
- elm_ridge_relevant_strava_edges.csv
- elm_ridge_relevant_strava_edges.png
- hourly_best_strava_model.png
- hourly_observed_vs_predicted.png
- hourly_test_predictions.csv
- monthly_best_strava_model.png
- monthly_observed_vs_predicted.png
- monthly_test_predictions.csv
- multi_frequency_model_results.csv
- strava_enhanced_model_results.csv
- strava_enhanced_test_predictions.csv
- strava_improvement_summary.csv
- strava_pca_coverage.csv
- test_coverage_summary.csv
- yearly_best_strava_model.png
- yearly_observed_vs_predicted.png
- yearly_test_predictions.csv
